# FineTuning



## Environment and paths


In [ ]:
# ============================================================
# 0. IMPORTS AND CONFIGURATION
# ============================================================
from pathlib import Path
import os
import json
import shutil
import random
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("Device:", device)

# Input directory
FT_RAW_DIR = "cartella_ft_14cm_raw"
FT_AUG_DIR = "cartella_ft_14cm_aug"

# Dataloader
BATCH_SIZE = 128
VAL_FRACTION = 0.15
NUM_WORKERS = 0

# Fine tuning
NUM_EPOCHS = 15
FINETUNE_LR = 1e-4
PHASE2_LR = 3e-5
UNFREEZE_EPOCH = 8

ALPHA_MA = 0.01
BETA_THICK = 1.0


# pre-training checkpoint path 
PRETRAIN_CKPT = "best_checkpoint_pretrain_T10_FULL_all.pth"

# best fine tuning model, checkpoint and loss log path
BEST_MODEL_PATH = "best_model_ft_T10_FULL_all.pth"
BEST_CKPT_PATH = "best_checkpoint_ft_T10_FULL_all.pth"
CSV_LOG_PATH = "train_val_ft_T10_FULL_all.csv"

PRINT_EVERY = 10

## Dataset implementation


In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset


class XRFLayerDataset(Dataset):
    """XRFLayerDataset: perform the processing step described by the implementation. See function arguments and returned values for details."""

    def __init__(
        self,
        file_paths,
        elements=None,
        add_org=True,
        normalize="max",  # "max", "sum" oppure None
    ):
        super().__init__()
        self.file_paths = sorted(file_paths)
        self.add_org = add_org
        self.normalize = normalize

        # Input/output file handling.
        if elements is None:
            elements = sorted(self._infer_elements())
        self.elements = elements
        self.elem_to_idx = {e: i for i, e in enumerate(self.elements)}

        self.n_layers = 3
        self.n_elems = len(self.elements) + (1 if self.add_org else 0)  # + Org

    def _infer_elements(self):
        """_infer_elements: perform the processing step described by the implementation. See function arguments and returned values for details."""
        elems = set()
        for path in self.file_paths:
            with open(path, "r") as f:
                d = json.load(f)
            comp_dict = d["labels"]["composition"]
            for layer_name, layer_comp in comp_dict.items():
                for el in layer_comp.keys():
                    elems.add(el)
        return elems

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        with open(path, "r") as f:
            d = json.load(f)

        # --------------------------
        # Spectrum processing and normalization.
        # --------------------------
        counts = np.array(d["counts"], dtype=np.float32)  # [2048]

        if self.normalize is not None:
            if self.normalize == "max":
                m = counts.max()
                if m > 0:
                    counts = counts / m
            elif self.normalize == "sum":
                s = counts.sum()
                if s > 0:
                    counts = counts / s
            else:
                raise ValueError(f"normalize='{self.normalize}' not supported")

        x = torch.from_numpy(counts).unsqueeze(0)  # [1, 2048]

        # --------------------------
        # LABELS
        # --------------------------
        labels = d["labels"]

        # layer_present: [3] (0/1)
        layer_present = torch.tensor(labels["layer_present"], dtype=torch.float32)

        # thickness_rho_t: [3]
        thickness = torch.tensor(labels["thickness_rho_t"], dtype=torch.float32)

        # organic_frac: [3]
        organic = torch.tensor(labels["organic_frac"], dtype=torch.float32)

        # Elemental composition processing.
        comp_full = torch.zeros((self.n_layers, self.n_elems), dtype=torch.float32)

        for li in range(self.n_layers):
            lname = f"layer_{li+1}"

            # parte inorganica
            if lname in labels["composition"]:
                for el, val in labels["composition"][lname].items():
                    if el in self.elem_to_idx:
                        comp_full[li, self.elem_to_idx[el]] = float(val)

            # Implementation note for this processing step.
            if self.add_org:
                idx_org = len(self.elements)
                comp_full[li, idx_org] = float(organic[li])

            # Layer-wise processing and masking.
            if layer_present[li] > 0.5:
                # Normalize so that the sum equals 1
                s = comp_full[li].sum()
                if s > 0:
                    comp_full[li] = comp_full[li] / s
            else:
                # Layer-wise processing and masking.
                comp_full[li].zero_()
                if self.add_org:
                    comp_full[li, idx_org] = 1.0

        return x, layer_present, thickness, comp_full


from glob import glob


from glob import glob
import os

def build_xrf_dataset_from_dirs(
    *dirs,
    elements=None,
    add_org=True,
    normalize=None,
):
    """build_xrf_dataset_from_dirs: perform the processing step described by the implementation. See function arguments and returned values for details."""

    # Collect all JSON files from each specified directory
    all_files = []
    for d in dirs:
        files = sorted(glob(os.path.join(d, "*.json")))
        all_files.extend(files)

    if len(all_files) == 0:
        raise RuntimeError(f"Nessun file JSON trovato nelle directory: {dirs}")

    # Create the dataset (if elements=None, infer them by scanning files)
    dataset = XRFLayerDataset(
        file_paths=all_files,
        elements=elements,
        add_org=add_org,
        normalize=normalize,
    )

    return dataset, dataset.elements



from torch.utils.data import DataLoader

from torch.utils.data import random_split

def split_xrf_dataset(dataset, val_fraction=0.2, seed=42):
    """split_xrf_dataset: perform the processing step described by the implementation. See function arguments and returned values for details."""
    n_total = len(dataset)
    n_val = int(n_total * val_fraction)
    n_train = n_total - n_val

    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
        dataset, [n_train, n_val], generator=generator
    )
    return train_dataset, val_dataset

class CachedDataset(Dataset):
    """Materializza una volta l'output ESATTO del dataset originale in RAM."""
    def __init__(self, base_dataset, desc="Caching FT dataset"):
        self.base_dataset = base_dataset
        self.elements = getattr(base_dataset, "elements", None)
        self.cache = []
        print(f"Materializing dataset: {len(base_dataset)} samples...")
        for i in tqdm(range(len(base_dataset)), desc=desc, unit="sample"):
            item = base_dataset[i]
            self.cache.append(tuple(
                x.detach().clone() if torch.is_tensor(x) else x
                for x in item
            ))
        print("Cache RAM completata.")

    def __len__(self):
        return len(self.cache)

    def __getitem__(self, idx):
        return self.cache[idx]


## Fine-tuning data loaders


In [ ]:
import os
from glob import glob
import torch
from torch.utils.data import DataLoader, random_split

def build_finetuning_dataloaders(
    aug_dirs,              # augmented spectra directory
    raw_dirs,              # raw spectra directory
    elements,              # Element list from pre-training 
    batch_size=128,
    val_fraction=0.15,
    seed=42,
    normalize="max",
    add_org=True,
    num_workers=0,
):
    """build_finetuning_dataloaders: perform the processing step described by the implementation. See function arguments and returned values for details."""

    # ---- raccogli file ----
    all_files = []

    for d in aug_dirs:
        all_files.extend(sorted(glob(os.path.join(d, "*.json"))))

    for d in raw_dirs:
        all_files.extend(sorted(glob(os.path.join(d, "*.json"))))

    if len(all_files) == 0:
        raise RuntimeError("Nessun file JSON trovato")

    # ---- dataset ----
    dataset = XRFLayerDataset(
        file_paths=all_files,
        elements=elements,   # DO NOT infer again
        add_org=add_org,
        normalize=normalize,
    )

    # ---- split ----
    n_total = len(dataset)
    n_val = int(val_fraction * n_total)
    n_train = n_total - n_val

    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(
        dataset,
        [n_train, n_val],
        generator=generator
    )

    # ---- dataloader ----
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    print(f"Fine-tuning dataset:")
    print(f"  totale      : {n_total}")
    print(f"  train       : {n_train}")
    print(f"  validation  : {n_val}")

    return train_loader, val_loader, dataset

## Element vocabulary and model setup


In [ ]:
# ============================================================
# ELEMENTS: MUST MATCH PRE-TRAINING
# ============================================================
pre_ckpt = torch.load(
    PRETRAIN_CKPT,
    map_location="cpu"
)

if "elements" in pre_ckpt:
    elements = list(pre_ckpt["elements"])
    print("Elements letti dal checkpoint:", elements)
else:
    raise KeyError(
        "The checkpoint does not contain 'elements'. "
        "Imposta manualmente `elements = [...]` usando ESATTAMENTE "
        "la lista del pre-training prima di continuare."
    )

train_loader, val_loader, dataset_ft = build_finetuning_dataloaders(
    aug_dirs=[FT_AUG_DIR],
    raw_dirs=[FT_RAW_DIR],
    elements=elements,
    batch_size=BATCH_SIZE,
    val_fraction=VAL_FRACTION,
    seed=SEED,
    normalize="max",
    add_org=True,
    num_workers=NUM_WORKERS,
)

print("Elements:", elements)
print("Org index:", len(elements))

## Network architecture — `XRFMultilayerNetLocalComp`

The input is a **2048-channel XRF spectrum** represented as `[batch, 1, 2048]`.

1. **Shared 1D convolutional encoder:** four Conv1d blocks with channels `1 → 64 → 128 → 256 → 256`, kernel size 7, stride 2, padding 3, ReLU and convolutional dropout (0.05).
2. **Presence head:** predicts the logit for the presence of the second paint layer (`presence2_logit`, shape `[B]`).
3. **Thickness head:** predicts `log10(ρt)` for three stratigraphic positions (`thickness_log`, shape `[B, 3]`).
4. **Local composition head:** processes local spectral features with convolution, adaptive pooling (8 bins), and a shared projection; predicts per-layer elemental composition logits, including the organic/other category (`comp_logits`, shape `[B, 3, M]`, where `M = len(elements) + 1`).

The three outputs are trained jointly. The architecture is identical in pre-training and fine-tuning; the latter initializes its parameters from the pre-trained checkpoint.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class XRFMultilayerNetLocalComp(nn.Module):
    """
    T10 FULL architecture.

    Encoder HPO fixed:
        1 -> 64 -> 128 -> 256 -> 256
        kernel=7, stride=2

    Local composition branch T10:
        pool=8
        local_kernel=7
        local_channels=256 -> 128
        shared_hidden=256
        local_dropout=0.10

    Output:
        presence2_logit : [B]
        thickness_log   : [B, 3]
        comp_logits     : [B, 3, len(elements)+1]
    """
    def __init__(
        self,
        n_channels=1,
        spec_len=2048,
        elements=None,
        latent_dim=256,
        comp_pool_bins=8,
        comp_hidden=256,
        local_kernel=7,
        local_channels=256,
        local_dropout=0.10,
        drop_conv=0.05,
        drop_latent=0.10,
        drop_head=0.00,
    ):
        super().__init__()
        if elements is None:
            elements = []

        self.elements = list(elements)
        self.n_layers = 3
        self.n_elems = len(elements) + 1
        self.latent_dim = 256
        self.comp_pool_bins = int(comp_pool_bins)
        self.local_kernel = int(local_kernel)
        self.local_channels = int(local_channels)
        self.local_dropout = float(local_dropout)

        self.do_conv = nn.Dropout1d(p=drop_conv) if drop_conv > 0 else nn.Identity()
        self.do_latent = nn.Dropout(p=drop_latent) if drop_latent > 0 else nn.Identity()
        self.do_head = nn.Dropout(p=drop_head) if drop_head > 0 else nn.Identity()

        # T10 fixed encoder: base_channels=64, depth=4, k=7, all stride=2
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(64, 128, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(128, 256, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(256, 256, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,
        )

        self.head_presence2 = nn.Linear(256, 1)
        self.head_thickness_log = nn.Linear(256, self.n_layers)

        pad_local = self.local_kernel // 2
        local_hidden_2 = max(self.local_channels // 2, 1)

        comp_pre_layers = [
            nn.Conv1d(256, self.local_channels, kernel_size=self.local_kernel, padding=pad_local),
            nn.ReLU(inplace=True),
        ]
        if self.local_dropout > 0:
            comp_pre_layers.append(nn.Dropout1d(p=self.local_dropout))

        comp_pre_layers += [
            nn.Conv1d(self.local_channels, local_hidden_2, kernel_size=self.local_kernel, padding=pad_local),
            nn.ReLU(inplace=True),
        ]
        if self.local_dropout > 0:
            comp_pre_layers.append(nn.Dropout1d(p=self.local_dropout))

        self.comp_pre = nn.Sequential(*comp_pre_layers)
        self.comp_pool = nn.AdaptiveAvgPool1d(self.comp_pool_bins)

        comp_in_dim = local_hidden_2 * self.comp_pool_bins + 256
        self.comp_shared = nn.Sequential(
            nn.Linear(comp_in_dim, comp_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p=drop_head) if drop_head > 0 else nn.Identity(),
        )

        self.head_comp_l1 = nn.Linear(comp_hidden, self.n_elems)
        self.head_comp_l2 = nn.Linear(comp_hidden, self.n_elems)
        self.head_comp_l3 = nn.Linear(comp_hidden, self.n_elems)

    def forward(self, x):
        z_map = self.encoder(x)
        z_global = self.do_latent(z_map.mean(dim=-1))
        z_head = self.do_head(z_global)

        presence2_logit = self.head_presence2(z_head).squeeze(-1)
        thickness_log = self.head_thickness_log(z_head)

        z_comp_local = self.comp_pre(z_map)
        z_comp_local = self.comp_pool(z_comp_local).flatten(1)
        z_comp = torch.cat([z_comp_local, z_global], dim=1)
        z_comp = self.comp_shared(z_comp)

        comp_logits = torch.stack([
            self.head_comp_l1(z_comp),
            self.head_comp_l2(z_comp),
            self.head_comp_l3(z_comp),
        ], dim=1)

        return {
            "z_global": z_global,
            "z_map": z_map,
            "presence2_logit": presence2_logit,
            "thickness_log": thickness_log,
            "comp_logits": comp_logits,
        }


## Optimizer and learning rates


In [ ]:
# PRETRAINING lr
LR = 1e-3       

# Phase 1 lr.
FINETUNE_LR = 1e-4   

# Phase 2 lr.
PHASE2_LR = 3e-5    

WEIGHT_DECAY = 1e-5

# ============================================================
# MODEL INITIALIZATION 
# ============================================================
model = XRFMultilayerNetLocalComp(
    n_channels=1,
    spec_len=2048,
    elements=elements,
    latent_dim=256,
    comp_pool_bins=8,
    comp_hidden=256,
    local_kernel=7,
    local_channels=256,
    local_dropout=0.10,
    drop_conv=0.05,
    drop_latent=0.10,
    drop_head=0.00,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Initialized Model")
print(f"Total parameters: {n_params:,}")
print(f"Trainable Parameters: {n_trainable:,}")



## Multi-task loss — three original objectives

Let $B$ be the batch size, $L=3$ the number of stratigraphic positions, $M$ the number of composition categories, $m_{b,l}\in\{0,1\}$ the layer-presence mask, and $N_{\mathrm{valid}}=\sum_{b,l}m_{b,l}$.

**1. Layer-presence loss (binary cross-entropy with logits)**, applied to the **second layer only**:

$$
\mathcal{L}_{\mathrm{presence}} =
-\frac{1}{B}\sum_{b=1}^{B}
\left[
y_{b,2}\log\sigma(z_b)
+(1-y_{b,2})\log(1-\sigma(z_b))
\right].
$$

**2. Thickness loss (masked mean squared error)**, using the logarithm of the *areal density* $\rho t$ for present layers:

$$
\mathcal{L}_{\mathrm{thickness}} =
\frac{1}{N_{\mathrm{valid}}}
\sum_{b=1}^{B}\sum_{l=1}^{L}m_{b,l}
\left[
\widehat{\log_{10}(\rho t)}_{b,l}
-\log_{10}((\rho t)_{b,l})
\right]^2.
$$

**3. Composition loss (masked soft-label cross-entropy)**:

$$
\mathcal{L}_{\mathrm{composition}} =
-\frac{1}{N_{\mathrm{valid}}+\varepsilon}
\sum_{b=1}^{B}\sum_{l=1}^{L}m_{b,l}
\sum_{j=1}^{M}c_{b,l,j}
\log\left[
\operatorname{softmax}(z^{(c)}_{b,l})_j
\right].
$$

The function below returns these three **raw** losses separately. The training loop combines them using the original moving-average normalization and thickness weighting; it is **not** a simple unweighted sum.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def xrf_multilayer_loss_raw(outputs,
                            layer_present,
                            thickness,
                            comp_true,
                            eps=1e-8):
    """Return the three raw multi-task losses: binary presence BCE for layer 2, masked MSE of log10 areal density, and masked soft-label composition cross-entropy."""
    presence2_logit = outputs["presence2_logit"]      # [B]
    thickness_log_pred = outputs["thickness_log"]     # [B, 3]
    comp_logits = outputs["comp_logits"]              # [B, 3, M]

    device = presence2_logit.device

    # 1) Presence (BCE) sul layer_2
    target_layer2 = layer_present[:, 1]  # [B]
    bce = nn.BCEWithLogitsLoss()
    loss_presence = bce(presence2_logit, target_layer2)

    # Layer-wise processing and masking.
    mask = (layer_present > 0.5)  # [B,3] bool
    valid_thickness = thickness[mask]                  # [N_valid]
    valid_thickness_log_pred = thickness_log_pred[mask]  # [N_valid]

    if valid_thickness.numel() > 0:
        valid_thickness_log_true = torch.log10(valid_thickness)
        loss_thickness = F.mse_loss(valid_thickness_log_pred,
                                    valid_thickness_log_true)
    else:
        loss_thickness = torch.tensor(0.0, device=device)

    # Layer-wise processing and masking.
    log_probs = F.log_softmax(comp_logits, dim=-1)       # [B,3,M]
    ce_per_layer = -(comp_true * log_probs).sum(dim=-1)  # [B,3]

    ce_masked = ce_per_layer * mask.float()
    if mask.sum() > 0:
        loss_comp = ce_masked.sum() / (mask.sum() + eps)
    else:
        loss_comp = torch.tensor(0.0, device=device)

    return loss_presence, loss_thickness, loss_comp

## Training


In [ ]:
from tqdm.auto import tqdm
import torch


def train_one_epoch(
    model,
    train_loader,
    optimizer,
    device,
    epoch=None,
    ma_pres=1.0,
    ma_thick=1.0,
    ma_comp=1.0,
    alpha_ma=0.01,
    beta_thick=1.0
):
    model.train()  # dropout ON

    running_loss_norm = 0.0
    running_pres = 0.0
    running_thick = 0.0
    running_comp = 0.0
    running_raw = 0.0
    n_batches = 0

    desc = f"Epoch {epoch} [train]" if epoch is not None else "Train"

    pbar = tqdm(
        train_loader,
        desc=desc,
        leave=True,
        dynamic_ncols=True,
        disable=False,
    )

    for x, layer_present, thickness, comp_full in pbar:

        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)
        comp_full = comp_full.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)

        loss_pres, loss_thick, loss_comp = xrf_multilayer_loss_raw(
            outputs,
            layer_present,
            thickness,
            comp_full
        )

        # Moving averages
        ma_pres = (
            (1 - alpha_ma) * ma_pres
            + alpha_ma * float(loss_pres.detach().item())
        )

        ma_thick = (
            (1 - alpha_ma) * ma_thick
            + alpha_ma * float(loss_thick.detach().item())
        )

        ma_comp = (
            (1 - alpha_ma) * ma_comp
            + alpha_ma * float(loss_comp.detach().item())
        )

        # Loss normalizzata usata per backward
        loss_norm = (
            loss_pres / (ma_pres + 1e-8)
            + beta_thick * loss_thick / (ma_thick + 1e-8)
            + loss_comp / (ma_comp + 1e-8)
        )

        loss_raw = (
            loss_pres
            + loss_thick
            + loss_comp
        )

        loss_norm.backward()
        optimizer.step()

        # Accumulatori
        running_loss_norm += float(loss_norm.detach().item())
        running_pres += float(loss_pres.detach().item())
        running_thick += float(loss_thick.detach().item())
        running_comp += float(loss_comp.detach().item())
        running_raw += float(loss_raw.detach().item())

        n_batches += 1

        # ====================================================
        # Implementation note for this processing step.
        # ====================================================
        pbar.set_postfix(
            loss=f"{running_loss_norm / n_batches:.4f}",
            pres=f"{running_pres / n_batches:.4f}",
            thick=f"{running_thick / n_batches:.4f}",
            comp=f"{running_comp / n_batches:.4f}",
        )

    metrics = {
        "loss_norm": running_loss_norm / max(n_batches, 1),
        "loss_raw": running_raw / max(n_batches, 1),
        "presence": running_pres / max(n_batches, 1),
        "thickness": running_thick / max(n_batches, 1),
        "composition": running_comp / max(n_batches, 1),
        "ma_pres": ma_pres,
        "ma_thick": ma_thick,
        "ma_comp": ma_comp,
    }

    return metrics, ma_pres, ma_thick, ma_comp

## Validation


In [ ]:
@torch.no_grad()
def validate_one_epoch(
    model,
    val_loader,
    device,
    epoch=None,
):
    model.eval()  # dropout OFF

    running_raw = 0.0
    running_pres = 0.0
    running_thick = 0.0
    running_comp = 0.0
    n_batches = 0

    desc = f"Epoch {epoch} [val]" if epoch is not None else "Validation"

    pbar = tqdm(
        val_loader,
        desc=desc,
        leave=True,
        dynamic_ncols=True,
        disable=False,
    )

    for x, layer_present, thickness, comp_full in pbar:

        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)
        comp_full = comp_full.to(device)

        outputs = model(x)

        loss_pres, loss_thick, loss_comp = xrf_multilayer_loss_raw(
            outputs,
            layer_present,
            thickness,
            comp_full
        )

        loss_raw = (
            loss_pres
            + loss_thick
            + loss_comp
        )

        running_raw += float(loss_raw.detach().item())
        running_pres += float(loss_pres.detach().item())
        running_thick += float(loss_thick.detach().item())
        running_comp += float(loss_comp.detach().item())

        n_batches += 1

        # ====================================================
        # Implementation note for this processing step.
        # ====================================================
        pbar.set_postfix(
            loss=f"{running_raw / n_batches:.4f}",
            pres=f"{running_pres / n_batches:.4f}",
            thick=f"{running_thick / n_batches:.4f}",
            comp=f"{running_comp / n_batches:.4f}",
        )

    metrics = {
        "loss_raw": running_raw / max(n_batches, 1),
        "presence": running_pres / max(n_batches, 1),
        "thickness": running_thick / max(n_batches, 1),
        "composition": running_comp / max(n_batches, 1),
    }

    return metrics

## Presence and composition metrics


In [ ]:
@torch.no_grad()
def eval_presence_accuracy(model, data_loader, device, threshold=0.5):
    model.eval()
    correct = 0
    total = 0
    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        outputs = model(x)
        probs = torch.sigmoid(outputs["presence2_logit"])  # [B]
        preds = (probs > threshold).float()
        target = layer_present[:, 1]  # Layer-wise processing and masking.
        correct += (preds == target).float().sum().item()
        total += target.numel()
    return correct / total

@torch.no_grad()
def eval_thickness_mae(model, data_loader, device):
    model.eval()
    abs_errors = []
    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)

        outputs = model(x)
        th_log_pred = outputs["thickness_log"]  # [B,3]
        th_pred = 10.0 ** th_log_pred           # [B,3]

        mask = (layer_present > 0.5)
        if mask.any():
            err = torch.abs(th_pred - thickness)[mask]
            abs_errors.append(err.detach().cpu())

    if len(abs_errors) == 0:
        return None
    abs_errors = torch.cat(abs_errors, dim=0)
    return abs_errors.mean().item(), abs_errors.median().item()

@torch.no_grad()
def eval_thickness_relative_error(model, data_loader, device, eps=1e-12):
    model.eval()
    rel_errors = []

    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)

        outputs = model(x)
        t_log_pred = outputs["thickness_log"]  # [B,3]
        t_pred = 10 ** t_log_pred              # [B,3]

        mask = (layer_present > 0.5)

        if mask.any():
            # Relative error: |pred - true| / true
            rel = torch.abs(t_pred - thickness) / (thickness + eps)
            rel = rel[mask]
            rel_errors.append(rel.detach().cpu())

    if len(rel_errors) == 0:
        return None, None

    rel_errors = torch.cat(rel_errors, dim=0)
    return rel_errors.mean().item(), rel_errors.median().item()


import numpy as np
import torch
import torch.nn.functional as F

@torch.no_grad()
def eval_composition_metrics(
    model,
    data_loader,
    device,
    org_index=None,
    org_min_true=1e-3,
    topk=(1, 3),
):
    """eval_composition_metrics: perform the processing step described by the implementation. See function arguments and returned values for details."""

    model.eval()

    l1_list = []
    l2_list = []
    kl_list = []

    top1_correct = 0
    top3_correct = 0
    total_layers = 0

    org_abs_errors = []
    org_rel_errors = []

    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        comp_full = comp_full.to(device)

        outputs = model(x)
        comp_logits = outputs["comp_logits"]          # [B, 3, M]
        comp_pred = F.softmax(comp_logits, dim=-1)    # [B, 3, M]

        # Layer-wise processing and masking.
        mask = (layer_present > 0.5)                  # [B, 3]
        if mask.sum() == 0:
            continue

        # Layer-wise processing and masking.
        true_flat = comp_full[mask]                   # [N_layers, M]
        pred_flat = comp_pred[mask]                   # [N_layers, M]

        # normalizziamo per sicurezza (piccoli drift numerici)
        true_flat = true_flat / (true_flat.sum(dim=-1, keepdim=True) + 1e-8)
        pred_flat = pred_flat / (pred_flat.sum(dim=-1, keepdim=True) + 1e-8)

        # Layer-wise processing and masking.
        l1 = torch.sum(torch.abs(pred_flat - true_flat), dim=-1)          # [N_layers]
        l2 = torch.sqrt(torch.sum((pred_flat - true_flat) ** 2, dim=-1))  # [N_layers]

        l1_list.append(l1.cpu().numpy())
        l2_list.append(l2.cpu().numpy())

        # Implementation note for this processing step.
        eps = 1e-8
        mask_pos = (true_flat > 0)
        kl = torch.sum(
            torch.where(
                mask_pos,
                true_flat * (torch.log(true_flat + eps) - torch.log(pred_flat + eps)),
                torch.zeros_like(true_flat),
            ),
            dim=-1,
        )  # [N_layers]
        kl_list.append(kl.cpu().numpy())

        # --------------------------------------------------------
        # Top-1 and Top-3 for dominant elements
        # Implementation note for this processing step.
        # --------------------------------------------------------
        if org_index is not None:
            # Dataset preparation.
            true_inorg = true_flat[:, :org_index]   # [N_layers, M_inorg]
            pred_inorg = pred_flat[:, :org_index]

            true_top1 = torch.argmax(true_inorg, dim=-1)   # [N_layers]
            pred_top1 = torch.argmax(pred_inorg, dim=-1)   # [N_layers]
        else:
            # Implementation note for this processing step.
            true_top1 = torch.argmax(true_flat, dim=-1)
            pred_top1 = torch.argmax(pred_flat, dim=-1)

        top1_correct += (true_top1 == pred_top1).sum().item()

        # top-3 overlap
        if 3 in topk:
            if org_index is not None:
                k = min(3, org_index)  # se pochi inorganici, riduci k
                true_topk = torch.topk(true_inorg, k=k, dim=-1).indices  # [N_layers, k]
                pred_topk = torch.topk(pred_inorg, k=k, dim=-1).indices  # [N_layers, k]
            else:
                k = 3
                true_topk = torch.topk(true_flat, k=k, dim=-1).indices
                pred_topk = torch.topk(pred_flat, k=k, dim=-1).indices

            # Layer-wise processing and masking.
            for i in range(true_flat.size(0)):
                set_true = set(true_topk[i].tolist())
                set_pred = set(pred_topk[i].tolist())
                if len(set_true.intersection(set_pred)) > 0:
                    top3_correct += 1

        total_layers += true_flat.size(0)

        # Implementation note for this processing step.
        if org_index is not None:
            org_true = true_flat[:, org_index]  # [N_layers]
            org_pred = pred_flat[:, org_index]

            abs_err = torch.abs(org_pred - org_true)
            org_abs_errors.append(abs_err.cpu().numpy())

            # Implementation note for this processing step.
            mask_org = (org_true > org_min_true)
            if mask_org.any():
                rel_err = (abs_err[mask_org] / org_true[mask_org])
                org_rel_errors.append(rel_err.cpu().numpy())

    if total_layers == 0:
        return {}

    # concatena tutto
    l1_all = np.concatenate(l1_list, axis=0)
    l2_all = np.concatenate(l2_list, axis=0)
    kl_all = np.concatenate(kl_list, axis=0)

    metrics = {
        "num_layers": int(total_layers),
        "l1_mean": float(l1_all.mean()),
        "l1_median": float(np.median(l1_all)),
        "l2_mean": float(l2_all.mean()),
        "l2_median": float(np.median(l2_all)),
        "kl_mean": float(kl_all.mean()),
        "top1_acc": top1_correct / total_layers,
        "top3_acc": top3_correct / total_layers if total_layers > 0 else 0.0,
    }

    if org_index is not None and len(org_abs_errors) > 0:
        org_abs_all = np.concatenate(org_abs_errors, axis=0)
        metrics["org_mae"] = float(org_abs_all.mean())

        if len(org_rel_errors) > 0:
            org_rel_all = np.concatenate(org_rel_errors, axis=0)
            metrics["org_rel_mean"] = float(org_rel_all.mean())
            metrics["org_rel_median"] = float(np.median(org_rel_all))

    return metrics

## Composition report


In [ ]:
def print_composition_report(metrics):
    """print_composition_report: perform the processing step described by the implementation. See function arguments and returned values for details."""

    if not metrics:
        print("No layer evaluated.")
        return

    print("=== Composition metrics ===")
    print(f"- Num. layer valutati      : {metrics['num_layers']}")

    print("\n> Distribution errors (all elements, including Org)")
    print(f"  L1 mean                  : {metrics['l1_mean']:.4f} "
          f"(sum |pred-true| mean per layer)")
    print(f"  L1 median                : {metrics['l1_median']:.4f} "
          f"(median)")
    print(f"  L2 mean                  : {metrics['l2_mean']:.4f} "
          f"(RMSE over composition per layer)")
    print(f"  L2 median                : {metrics['l2_median']:.4f}")
    print(f"  KL mean (true || pred)   : {metrics['kl_mean']:.4f} ")

    print("\n> Dominant elements")
    print(f"  Top-1 accuracy           : {metrics['top1_acc']:.3f} ")
    print(f"  Top-3 accuracy           : {metrics['top3_acc']:.3f} ")

    if "org_mae" in metrics:
        print("\n> Organic component (Org)")
        print(f"  Org MAE                  : {metrics['org_mae']:.4f} "
              f"(mean absolute error in the organic fraction)")
        if "org_rel_mean" in metrics:
            print(f"  Org REL mean             : {metrics['org_rel_mean']:.3f} "
                  f"(mean relative error for Org, only where Org_true>{1e-3})")
            print(f"  Org REL median           : {metrics['org_rel_median']:.3f} "
                  f"(median relative error for Org)")


import matplotlib.pyplot as plt
import numpy as np

def plot_loss_curves(train_losses, val_losses, title="Loss curves"):
    """
    Plot loss curve per epoch:
    """
    train_losses = np.asarray(train_losses, dtype=float)
    val_losses = np.asarray(val_losses, dtype=float)

    epochs = np.arange(1, len(train_losses) + 1)

    plt.figure(figsize=(7, 5))
    plt.plot(epochs, train_losses, marker="o", label="Train loss (norm)")
    plt.plot(epochs, val_losses, marker="o", label="Val loss (raw)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## CSV logging


In [ ]:
import os
import csv

CSV_LOG_PATH = "train_val_ft_T10.csv"

def log_epoch_to_csv(csv_path, epoch, train_metrics, val_metrics):
    """log_epoch_to_csv: perform the processing step described by the implementation. See function arguments and returned values for details."""
    file_exists = os.path.exists(csv_path)

    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow([
                "epoch",
                "train_loss_raw",
                "train_loss_norm",
                "train_loss_pres",
                "train_loss_thick",
                "train_loss_comp",
                "val_loss_raw",
                "val_loss_pres",
                "val_loss_thick",
                "val_loss_comp",
            ])

        writer.writerow([
            epoch,
            train_metrics["loss_raw"],
            train_metrics["loss_norm"],
            train_metrics["presence"],
            train_metrics["thickness"],
            train_metrics["composition"],
            val_metrics["loss_raw"],
            val_metrics["presence"],
            val_metrics["thickness"],
            val_metrics["composition"],
        ])

## Freeze/unfreeze 


In [ ]:
def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


def freeze_encoder_train_heads(model):
    """freeze_encoder_train_heads: perform the processing step described by the implementation. See function arguments and returned values for details."""
    # congela encoder
    set_requires_grad(model.encoder, False)

    # sblocca head globali
    set_requires_grad(model.head_presence2, True)
    set_requires_grad(model.head_thickness_log, True)

    # Elemental composition processing.
    set_requires_grad(model.comp_pre, True)
    set_requires_grad(model.comp_shared, True)
    set_requires_grad(model.head_comp_l1, True)
    set_requires_grad(model.head_comp_l2, True)
    set_requires_grad(model.head_comp_l3, True)

    return model


def unfreeze_last_encoder_conv(model):
    """unfreeze_last_encoder_conv: perform the processing step described by the implementation. See function arguments and returned values for details."""
    set_requires_grad(model.encoder[9], True)
    return model


def count_trainable_params(model):
    n_tensors = sum(p.requires_grad for p in model.parameters())
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return n_tensors, n_params

## Trainable parameters


In [ ]:
def print_trainable_summary(model, verbose=True):
    n_tensors = 0
    n_params = 0
    trainables = []

    for name, p in model.named_parameters():
        if p.requires_grad:
            n_tensors += 1
            n_params += p.numel()
            trainables.append((name, p.numel()))

    print(
        f"Trainable tensors / trainable params: "
        f"({n_tensors}, {n_params})"
    )

    if verbose:
        for name, n in trainables:
            print(f"  {name}: {n:,}")

## Load pretrained weights


In [ ]:
# ============================================================
# LOAD PRETRAINING
# ============================================================
checkpoint = torch.load(
    PRETRAIN_CKPT,
    map_location=device
)

if "model_state" in checkpoint:
    model.load_state_dict(checkpoint["model_state"])
else:
    model.load_state_dict(checkpoint)

model.to(device)

print("Loaded pretraining checkpoint:", PRETRAIN_CKPT)

model = freeze_encoder_train_heads(model)
print_trainable_summary(model, verbose=False)

optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=FINETUNE_LR,
)

start_epoch = 1
best_val_loss = float("inf")

ma_pres = 1.0
ma_thick = 1.0
ma_comp = 1.0

train_loss_history = []
val_loss_history = []

did_unfreeze = False
org_index = len(elements)

print(
    f"Fine tuning: {NUM_EPOCHS} epochs | "
    f"unfreeze last conv @ {UNFREEZE_EPOCH}"
)

## Fine-tuning loop


In [ ]:
# ============================================================
# FINE-TUNING LOOP
# ============================================================

for epoch in range(start_epoch, NUM_EPOCHS + 1):

    if (
        not did_unfreeze
        and epoch == UNFREEZE_EPOCH
    ):
        model = unfreeze_last_encoder_conv(model)

        print("\n[Phase2] Unfreeze of last convolutional layer of the encoder.")
        print_trainable_summary(model, verbose=False)

        optimizer = torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad],
            lr=PHASE2_LR,
        )

        did_unfreeze = True

    # --------------------------------------------------------
    # TRAIN -> dropout ON
    # --------------------------------------------------------
    train_metrics, ma_pres, ma_thick, ma_comp = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        device=device,
        epoch=epoch,
        ma_pres=ma_pres,
        ma_thick=ma_thick,
        ma_comp=ma_comp,
        alpha_ma=ALPHA_MA,
        beta_thick=BETA_THICK,
    )

    # --------------------------------------------------------
    # VALIDATION -> dropout OFF
    # --------------------------------------------------------
    val_metrics = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        device=device,
        epoch=epoch,
    )

    train_loss_history.append(train_metrics["loss_raw"])
    val_loss_history.append(val_metrics["loss_raw"])

    if epoch % PRINT_EVERY == 0 or epoch == NUM_EPOCHS:
        print(
            f"\n[Epoch {epoch:03d}] "
            f"Train norm={train_metrics['loss_norm']:.4f} | "
            f"raw={train_metrics['loss_raw']:.4f} "
            f"(pres={train_metrics['presence']:.4f}, "
            f"thick={train_metrics['thickness']:.4f}, "
            f"comp={train_metrics['composition']:.4f}) | "
            f"Val raw={val_metrics['loss_raw']:.4f} "
            f"(pres={val_metrics['presence']:.4f}, "
            f"thick={val_metrics['thickness']:.4f}, "
            f"comp={val_metrics['composition']:.4f})"
        )

        print(
            f"    MA: pres={ma_pres:.5f} | "
            f"thick={ma_thick:.5f} | "
            f"comp={ma_comp:.5f}"
        )

    log_epoch_to_csv(
        CSV_LOG_PATH,
        epoch,
        train_metrics,
        val_metrics,
    )

    # --------------------------------------------------------
    # BEST
    # --------------------------------------------------------
    val_loss = val_metrics["loss_raw"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH,
        )

        if epoch % PRINT_EVERY == 0 or epoch == NUM_EPOCHS:
            print(
                f"    -> NUOVO BEST epoch {epoch}: "
                f"{best_val_loss:.6f}"
            )

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "phase2_unfreeze_done": did_unfreeze,

            "ma_pres": ma_pres,
            "ma_thick": ma_thick,
            "ma_comp": ma_comp,

            "alpha_ma": ALPHA_MA,
            "beta_thick": BETA_THICK,

            "finetune_lr": FINETUNE_LR,
            "phase2_lr": PHASE2_LR,
            "unfreeze_epoch": UNFREEZE_EPOCH,

            "elements": elements,
            "ft_air_dir": FT_AIR_DIR,
        },
        BEST_CKPT_PATH,
    )

print("\nFine-tuning ended.")
print("Best val loss:", best_val_loss)
print("Best model:", BEST_MODEL_PATH)
print("Checkpoint:", BEST_CKPT_PATH)

plot_loss_curves(
    train_loss_history,
    val_loss_history,
    title=(
        "FT model | "
        "encoder frozen -> heads -> last conv"
    ),
)